In [ ]:
!pip install langchain_groq

In [ ]:
!pip install dotenv

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
import os

# ✅ Paste your Groq API key here
os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"

In [ ]:
load_dotenv()

# Using llama3-8b-8192 via Groq — you can also use "llama3-70b-8192", "mixtral-8x7b-32768", etc.
model = ChatGroq(model="llama3-8b-8192")

In [ ]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    rating: int

In [ ]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [ ]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [ ]:
def rate_content(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']
    content = state['content']

    prompt = f'Compare the generated content with the outline and rate the - {content} as per the outline - \n {outline}'

    rate_content = model.invoke(prompt).content

    state['rating'] = rate_content

    return state

In [ ]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('rate_content', rate_content)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'rate_content')
graph.add_edge('rate_content', END)

workflow = graph.compile()


In [ ]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

In [ ]:
print(final_state['outline'])

In [ ]:
print(final_state['content'])

In [ ]:
print(final_state['rating'])